In [ ]:
import pandas as pd
import os

In [ ]:
cwd= os.getcwd()
print(cwd)
BASE_DIR= os.path.join(cwd,"..")

data_acd= os.path.join(BASE_DIR, "data", "acd.xlsx")
data_thalass= os.path.join(BASE_DIR, "data", "thalass.xlsx")
mix= os.path.join(BASE_DIR, "data", "data.xlsx")

### ACD

In [ ]:
df_acd= pd.read_excel(data_acd)
print(df_acd.head())
print(df_acd.shape)

In [ ]:
df_acd.info()

In [ ]:
df_acd = df_acd.drop(["Mã bệnh nhân"], axis=1)

df_acd = df_acd.rename(columns={
    "Họ và tên": "Giới",
    "Hb (g/l)": "Hb"
    })

original = df_acd["Chẩn đoán"]

# B2: overwrite toàn bộ thành ACD
df_acd["Chẩn đoán"] = "ACD"

# B3: thêm điều kiện dựa trên dữ liệu gốc
df_acd.loc[
    original.str.contains("thiếu máu thiếu sắt", case=False, na=False),
    "Chẩn đoán"
] += "+IDA"

df_acd.loc[
    original.str.contains("beta thalassemia", case=False, na=False),
    "Chẩn đoán"
] += "+Beta thalassemia"

df_acd.loc[
    original.str.contains("alpha thalassemia", case=False, na=False),
    "Chẩn đoán"
] += "+Alpha thalassemia"


print(df_acd.head())

### Thalassemia

In [ ]:
df_thalass= pd.read_excel(data_thalass)
print(df_thalass.shape)

In [ ]:
print(df_thalass.info())

In [ ]:
# Bước 1: lấy row 1 làm tên cột
df_thalass.columns = df_thalass.iloc[1]

# Bước 2: xóa 2 dòng đầu (row 0 và row 1)
df_thalass = df_thalass.drop([0, 1])

In [ ]:
df_thalass = df_thalass.iloc[:, 1:-1]
df_thalass = df_thalass.drop(["PID"], axis=1)
df_thalass = df_thalass.reset_index(drop=True)

In [ ]:
df_thalass = df_thalass.rename(columns={
    "RBC (T/l)": "RBC",
    "Hb (g/L)": "Hb",
    "Transferin (mg/dL)": "Transferin",
    "Đột biến gen thalassemia (nếu có)": "Đột biến gen thalassemia"
    
    })

In [ ]:
print(df_thalass.head())

In [ ]:
cols = ["RBC", "Hb", "MCV", "MCHC", "RDW-CV", "Fe", "Ferritin", "Transferin", "TIBC", "CRP", 
        "Ret-He", "HbA1", "HbA2", "HbF", "HbH", "HbBart", "HbS", "HbE", "Hb khác", "Tuổi"]



df_thalass[cols] = df_thalass[cols].apply(pd.to_numeric, errors="coerce")
df_thalass["Đột biến gen thalassemia"]= 1
print(df_thalass.head())

In [ ]:
df_part = df_thalass.head(15)
df_part = df_part.drop([10])
print(df_part)

### File hỗn độn

In [ ]:
df_mix= pd.read_excel(mix)
print(df_mix.shape)

In [ ]:
def clean_df(df):
    a= df.drop(index= range(3))
    b= a.reset_index(drop= True)
    b.columns= b.iloc[0]
    c= b.drop(index= 0)
    d= c.drop(columns= "STT")
    e = d.iloc[:, :-4] # bỏ cột tên NaN
    return e

cleaned_mix_df= clean_df(df_mix)
print(cleaned_mix_df.shape)

In [ ]:
text_cols = [1, 2, 23]

for i in range(len(cleaned_mix_df.columns)):
    if i not in text_cols:
        col_name = cleaned_mix_df.columns[i]
        # Gán lại bằng TÊN cột, không dùng iloc
        cleaned_mix_df[col_name] = pd.to_numeric(cleaned_mix_df[col_name], errors='coerce')
    else:
        col_name = cleaned_mix_df.columns[i]
        cleaned_mix_df[col_name] = cleaned_mix_df[col_name].astype("string")


cleaned_mix_df["Tiền sử hoặc bệnh kèm theo"] = cleaned_mix_df["Tiền sử hoặc bệnh kèm theo"].notna().astype(int)

print(cleaned_mix_df.info())

In [ ]:
cleaned_mix_df = cleaned_mix_df.rename(columns={
    "RBC (T/l)\nSố lượng hồng cầu": "RBC",
    "Hb (g/L)\nNồng độ Hemoglobin": "Hb",
    "MCV\nKích thước Hồng cầu": "MCV",
    "MCHC\nNồng độ hemoglogin trung bình hồng cầu": "MCHC",
    "RDW-CV\nĐộ phân bố kích thước hồng cầu": "RDW-CV",
    "Fe \nSắt huyết thanh": "Fe",
    "Ferritin\nDự trữ sắt": "Ferritin",
    "Transferin (mg/dL)\nProtein vận chuyển sắt": "Transferin",
    "TIBC\nkhả năng gắn sắt toàn bộ": "TIBC",
    "CRP\nChỉ số viêm": "CRP",
    "Ret-He\nChỉ số hemoglobin của hồng cầu lưới": "Ret-He",
    "HbA1\nhemboglobin A1": "HbA1",
    "HbA2\nHemoglobin A2": "HbA2",
    "HbF\nHemoglobin F": "HbF",
    "HbH\nHemoblobin H": "HbH",
    "HbBart\nHemoglobin Bart": "HbBart",
    "HbS\nHemoglobin S": "HbS",
    "HbE\nhemoglobin E": "HbE",
    "Đột biến gen thalassemia (nếu có)": "Đột biến gen thalassemia",
    "Chẩn đoán (chỉ liên quan hồng cầu nhỏ: IDA, ACD, Thalassemia, CRNN)": "Chẩn đoán"
    })

print(cleaned_mix_df.info())

In [ ]:
print(cleaned_mix_df.columns)

In [ ]:
import numpy as np

cleaned_mix_df["TSAT (%)"] = np.where(
    (cleaned_mix_df["Fe"].notna()) & (cleaned_mix_df["Transferin"].notna()) & (cleaned_mix_df["Transferin"] != 0),
    cleaned_mix_df["Fe"] / cleaned_mix_df["Transferin"] * 100,
    np.nan
)

In [ ]:
df_new = pd.concat([cleaned_mix_df, df_acd, df_part], axis=0)
print(df_new.info())
print(df_new.head())

In [ ]:
df_new["ACD"] = df_new["Chẩn đoán"].str.contains("ACD").astype(int)
df_new["IDA"] = df_new["Chẩn đoán"].str.contains("IDA").astype(int)
df_new["Thal"] = df_new["Chẩn đoán"].str.contains("thalassemia", case=False).astype(int)

df_new = pd.get_dummies(df_new, columns=["Giới"], drop_first=True)
df_new.to_csv(os.path.join(BASE_DIR, "data", "concat_for_eda.csv"), index=False)

df_new= df_new.drop(columns= "Chẩn đoán")
print(df_new.info())

In [ ]:
y = df_new[["ACD", "IDA", "Thal"]]

counts = y.value_counts()

print(counts)

In [ ]:
df_new = df_new[~((df_new["ACD"] == 1) & (df_new["Thal"] == 1))]

In [ ]:
df_new.to_csv(os.path.join(BASE_DIR, "data", "concat.csv"), index=False)
